# Day 33 — Safe autocomplete and prediction loading repair

This notebook backs up dashboard files, repairs autocomplete selection and validates prediction inputs before showing loading state. It does not change navigation labels or model/backend inference code.


In [129]:
from pathlib import Path
from datetime import datetime
import shutil, subprocess, sys, re ,json

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / "django_api" / "manage.py").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook with the repository root as the working directory.")
APP = ROOT / "django_api" / "ipl_api"
DJANGO = ROOT / "django_api"

js_candidates = list(APP.rglob("dashboard_day25.js")) or [p for p in APP.rglob("*.js") if "dashboard" in p.name.lower()]
if not js_candidates:
    raise FileNotFoundError("No dashboard JavaScript file found under django_api/ipl_api.")
JS_PATH = js_candidates[0]

html_candidates = list(APP.rglob("dashboard_day25.html"))
if not html_candidates:
    raise FileNotFoundError("No dashboard_day25.html template found under django_api/ipl_api.")
TEMPLATE = html_candidates[0]
URLS = APP / "urls.py"

BACKUP = ROOT / "backups" / ("day33_fix_" + datetime.now().strftime("%Y%m%d_%H%M%S"))
for f in (JS_PATH, TEMPLATE, URLS):
    if f.exists():
        dest = BACKUP / f.relative_to(ROOT)
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(f, dest)

print("Repository:", ROOT)
print("JavaScript:", JS_PATH)
print("Template  :", TEMPLATE)
print("Backup    :", BACKUP)


Repository: D:\Git\Personal\ipl-player-performance
JavaScript: D:\Git\Personal\ipl-player-performance\django_api\ipl_api\static\ipl_api\js\dashboard_day25.js
Template  : D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Backup    : D:\Git\Personal\ipl-player-performance\backups\day33_fix_20261024_020205


In [130]:
# Find stray literal backslash-n tokens that sit at the END of a code line
# (after ; ) } { , ). A legitimate "\\n" inside a string ends with a quote, so it is not matched.
STRAY = re.compile(r"(?<=[;\)\}\{,])[ \t]*\\n[ \t]*$")

def find_stray(text):
    return [(i + 1, line) for i, line in enumerate(text.split("\n")) if STRAY.search(line)]

for path in (JS_PATH, TEMPLATE):
    hits = find_stray(path.read_text(encoding="utf-8"))
    print(f"{path.name}: {len(hits)} stray literal \\n token(s)")
    for no, line in hits:
        print(f"  line {no}: {line.strip()!r}")


dashboard_day25.js: 0 stray literal \n token(s)
dashboard_day25.html: 0 stray literal \n token(s)


In [131]:
def node_check(source, label):
    """Optional extra check. Returns (None, message) when Node.js is not installed."""
    node = shutil.which("node")
    if not node:
        return None, "Node.js not installed - skipped the optional parser check (the fix does not need it)."
    tmp = BACKUP / (label + ".js")
    tmp.parent.mkdir(parents=True, exist_ok=True)
    tmp.write_text(source, encoding="utf-8")
    r = subprocess.run([node, "--check", str(tmp)], capture_output=True, text=True)
    return r.returncode, (r.stdout + "\n" + r.stderr).strip()

def repair(text):
    return "\n".join(STRAY.sub("", line) for line in text.split("\n"))

# --- JavaScript ---
js_old = JS_PATH.read_text(encoding="utf-8")
js_new = repair(js_old)
if js_new != js_old:
    JS_PATH.write_text(js_new, encoding="utf-8", newline="")
    print("JavaScript: removed stray literal \\n token(s).")
else:
    print("JavaScript: nothing to change (already clean).")

# --- Template (inline <script> blocks could carry the same bug) ---
html_old = TEMPLATE.read_text(encoding="utf-8")
html_new = repair(html_old)
if html_new != html_old:
    TEMPLATE.write_text(html_new, encoding="utf-8", newline="")
    print("Template: removed stray literal \\n token(s).")
else:
    print("Template: nothing to change.")

# --- Verify ---
remaining = find_stray(JS_PATH.read_text(encoding="utf-8"))
print("Remaining stray tokens in JS:", len(remaining))
rc, msg = node_check(JS_PATH.read_text(encoding="utf-8"), "final_check")
print("Node parse check:", "PASSED" if rc == 0 else ("SKIPPED" if rc is None else "FAILED"))
if rc not in (0, None):
    print(msg)
    raise RuntimeError("JavaScript still has a syntax error. See parser output above.")
if rc is None:
    print(msg)

lines = JS_PATH.read_text(encoding="utf-8").split("\n")
print("\nLines 183-187 now:")
for i in range(182, min(187, len(lines))):
    print(f"{i+1:04}: {lines[i]}")


JavaScript: nothing to change (already clean).
Template: nothing to change.
Remaining stray tokens in JS: 0
Node parse check: SKIPPED
Node.js not installed - skipped the optional parser check (the fix does not need it).

Lines 183-187 now:


In [132]:
# Print actual selectors and routes; do not guess endpoint names.
html = TEMPLATE.read_text(encoding="utf-8")
js = JS_PATH.read_text(encoding="utf-8")
ids = re.findall(r'\bid=["\']([^"\']+)["\']', html, flags=re.I)
print("Relevant HTML IDs:")
for value in ids:
    if any(w in value.lower() for w in ("player", "bat", "bowl", "dl", "predict", "result", "eda", "overview", "json", "search")):
        print(" -", value)

print("\nRelevant route definitions:")
if URLS.exists():
    for line in URLS.read_text(encoding="utf-8").splitlines():
        if any(w in line.lower() for w in ("player", "search", "predict", "dashboard", "eda")):
            print(line.strip())

print("\nJS request URLs:")
for url in sorted(set(re.findall(r'["\'](/api/[^"\']+)["\']', js))):
    print(" -", url)

print("\nLoading/autocomplete/JSON markers currently present:")
for marker in ("Predicting", "spinner-border", "autocomplete", "playerOptions", "battingDLJson", "bowlingDLJson"):
    print(f"{marker}: {marker.lower() in js.lower() or marker.lower() in html.lower()}")


Relevant HTML IDs:
 - refreshOverview
 - overview
 - overviewCards
 - playersMetric
 - battingMetric
 - bowlingMetric
 - player-search
 - playerName
 - playerOptions
 - searchPlayer
 - playerResult
 - batting
 - battingPlayer
 - battingJson
 - runBattingML
 - battingMLResult
 - battingDLStatus
 - battingDLJson
 - battingDLResult
 - bowling
 - bowlingPlayer
 - bowlingJson
 - runBowlingML
 - bowlingMLResult
 - bowlingDLJson
 - metricsResult
 - eda
 - edaEndpoint
 - loadEDA
 - edaResult
 - batch
 - batchEndpoint
 - batchJson
 - runBatch
 - downloadBatch
 - batchResult
 - statusResult
 - ipl-player-suggestions

Relevant route definitions:
from .deep_learning_prediction_day31 import deep_learning_prediction
from .player_suggestions_day31 import player_suggestions
from .views import PlayerListAPIView, PlayerDetailAPIView
from .dashboard_views_day26 import dashboard_day25
from .eda_summary_day26 import eda_summary_day26
from .eda_views import EDABattingAPIView, EDABowlingAPIView, EDASeasonAPI

In [133]:
# Safe dashboard UI repair: validate first, show loading only for valid prediction requests.
js_text = JS_PATH.read_text(encoding="utf-8")
html_text = TEMPLATE.read_text(encoding="utf-8")

api_urls = sorted(set(re.findall(r'["\'](/api/[^"\']*(?:players|player)[^"\']*)["\']', js_text, flags=re.I)))
search_url = next((u for u in api_urls if "search" in u.lower() or "options" in u.lower()), None)
if not search_url:
    search_url = next((u for u in api_urls if "players/" in u.lower() and "predict" not in u.lower()), None)

start_marker = "/* DAY33_UI_LOADING_AUTOCOMPLETE_HELPER_START */"
end_marker = "/* DAY33_UI_LOADING_AUTOCOMPLETE_HELPER_END */"
clean_html = re.sub(
    r'<script>\s*/\* DAY33_UI_LOADING_AUTOCOMPLETE_HELPER_START \*/.*?/\* DAY33_UI_LOADING_AUTOCOMPLETE_HELPER_END \*/\s*</script>\s*',
    '', html_text, flags=re.S
)

helper = r"""
/* DAY33_UI_LOADING_AUTOCOMPLETE_HELPER_START */
(function () {
  if (window.__day33UiHelperInstalledV3) return;
  window.__day33UiHelperInstalledV3 = true;

  // ---------- Player autocomplete ----------
  const playerSearchUrl = __SEARCH_URL__;
  const normaliseNames = data => {
    const arr = Array.isArray(data) ? data :
      (data && (data.results || data.players || data.options || data.data)) || [];
    return arr.map(item => typeof item === "string" ? item :
      (item.player_name || item.name || item.full_name || item.player || item.label || ""))
      .filter(Boolean);
  };
  if (playerSearchUrl) {
    const inputs = [...document.querySelectorAll("input")].filter(input => {
      const context = [input.id, input.name, input.placeholder, input.getAttribute("aria-label"),
        input.parentElement?.querySelector("label")?.textContent].filter(Boolean).join(" ");
      return /player[\s_-]*name/i.test(context);
    });
    inputs.forEach(input => {
      if (input.dataset.day33AutocompleteV3) return;
      input.dataset.day33AutocompleteV3 = "1";
      const wrapper = document.createElement("div");
      wrapper.style.position = "relative";
      input.parentNode.insertBefore(wrapper, input);
      wrapper.appendChild(input);
      const list = document.createElement("div");
      list.hidden = true;
      list.setAttribute("role", "listbox");
      list.style.cssText = "position:absolute;z-index:99999;left:0;right:0;top:100%;max-height:220px;overflow:auto;background:#fff;border:1px solid #cbd5e1;border-radius:6px;box-shadow:0 4px 12px #0003";
      wrapper.appendChild(list);
      let timer = null, seq = 0, chosenValue = null;
      const close = () => { list.hidden = true; list.replaceChildren(); };
      const selectName = name => {
        clearTimeout(timer); seq++; chosenValue = name;
        input.value = name;
        input.dispatchEvent(new Event("input", {bubbles:true}));
        input.dispatchEvent(new Event("change", {bubbles:true}));
        close();
      };
      input.addEventListener("input", () => {
        const q = input.value.trim();
        if (chosenValue && q === chosenValue) { close(); return; }
        chosenValue = null;
        clearTimeout(timer);
        const thisSeq = ++seq;
        if (q.length < 2) { close(); return; }
        timer = setTimeout(async () => {
          try {
            const sep = playerSearchUrl.includes("?") ? "&" : "?";
            const res = await fetch(playerSearchUrl + sep + "search=" + encodeURIComponent(q), {
              headers: {"Accept":"application/json"}
            });
            if (!res.ok) throw new Error("Player search HTTP " + res.status);
            const data = await res.json();
            if (thisSeq !== seq || input.value.trim() !== q) return;
            const names = [...new Set(normaliseNames(data))]
              .filter(name => name.toLowerCase().includes(q.toLowerCase())).slice(0, 12);
            list.replaceChildren();
            names.forEach(name => {
              const option = document.createElement("button");
              option.type = "button"; option.setAttribute("role", "option");
              option.textContent = name;
              option.style.cssText = "display:block;width:100%;text-align:left;padding:9px 10px;border:0;background:#fff;color:#111827;cursor:pointer";
              option.addEventListener("pointerdown", e => { e.preventDefault(); selectName(name); });
              option.addEventListener("mouseenter", () => option.style.background = "#eaf2ff");
              option.addEventListener("mouseleave", () => option.style.background = "#fff");
              list.appendChild(option);
            });
            list.hidden = names.length === 0;
          } catch (err) { close(); console.warn("Player autocomplete failed:", err); }
        }, 250);
      });
      input.addEventListener("keydown", e => {
        if (e.key === "Escape") close();
        if (e.key === "ArrowDown" && !list.hidden) {
          const first = list.querySelector('[role="option"]');
          if (first) { e.preventDefault(); first.focus(); }
        }
      });
      input.addEventListener("blur", () => setTimeout(() => {
        if (!wrapper.contains(document.activeElement)) close();
      }, 160));
      document.addEventListener("pointerdown", e => { if (!wrapper.contains(e.target)) close(); });
    });
  } else {
    console.warn("Player autocomplete not installed: no player-search endpoint was found in dashboard JS.");
  }

  // ---------- Prediction validation/loading ----------
  const originalLabels = new WeakMap();
  const activeButtons = new WeakSet();
  function getCard(button) { return button.closest(".card") || button.parentElement; }
  function getFields(button) {
    const card = getCard(button);
    const inputs = [...(card?.querySelectorAll("input") || [])];
    const areas = [...(card?.querySelectorAll("textarea") || [])];
    const player = inputs.find(el => /player/i.test([el.id,el.name,el.placeholder,
      el.getAttribute("aria-label"),el.parentElement?.querySelector("label")?.textContent].filter(Boolean).join(" ")));
    const json = areas.find(el => /json|sequence|feature/i.test([el.id,el.name,
      el.parentElement?.querySelector("label")?.textContent].filter(Boolean).join(" ")));
    return {card, player, json};
  }
  function validJsonForButton(button, text) {
    if (!text || !text.trim()) return false;
    try {
      const obj = JSON.parse(text);
      const label = (button.textContent || "").toLowerCase();
      if (/dl/.test(label)) {
        const seq = obj.sequence || obj.input_sequence || obj.X;
        return Array.isArray(seq) && seq.length > 0;
      }
      return obj && typeof obj === "object" && Object.keys(obj).length > 0;
    } catch (_) { return false; }
  }
  function isDefaultExample(text) {
    const t = (text || "").replace(/\s/g, "").toLowerCase();
    return t === '{"feature_name":0}' ||
      (t.includes('"sequence":') && t.includes('[0,0]') && t.includes('"preprocessed":false'));
  }
  function showResult(button, message) {
    const card = getCard(button);
    if (!card) return;
    let out = card.querySelector("pre, .json-result, .prediction-result, [id$='Result'], [id$='Status']");
    if (!out) {
      out = document.createElement("pre");
      out.className = "prediction-result";
      out.style.cssText = "white-space:pre-wrap;background:#0f172a;color:#f8fafc;padding:16px;border-radius:8px;margin-top:12px;min-height:72px";
      button.insertAdjacentElement("afterend", out);
    }
    out.textContent = message;
    out.setAttribute("aria-live", "polite");
  }
  function restore(button, message) {
    if (!activeButtons.has(button)) return;
    activeButtons.delete(button);
    button.disabled = false;
    button.removeAttribute("aria-busy");
    button.textContent = originalLabels.get(button) || "Run prediction";
    if (message) showResult(button, message);
  }
  document.addEventListener("click", event => {
    const button = event.target.closest("button");
    if (!button) return;
    const label = (button.textContent || "").trim();
    if (!/^run\b/i.test(label) || !/prediction/i.test(label) || /predicting/i.test(label)) return;

    const {player, json} = getFields(button);
    const playerName = (player?.value || "").trim();
    const jsonText = (json?.value || "").trim();
    const isDL = /\bdl\b/i.test(label);
    const jsonProvided = !!jsonText && !isDefaultExample(jsonText) && validJsonForButton(button, jsonText);
    if (!playerName && !jsonProvided) {
      event.preventDefault(); event.stopImmediatePropagation();
      showResult(button, isDL
        ? "Enter a player name or provide a valid DL sequence JSON."
        : "Enter a player name or provide valid feature JSON.");
      return;
    }
    // Never show a loading state for a click that lacks usable input.
    originalLabels.set(button, label);
    activeButtons.add(button);
    button.disabled = true;
    button.setAttribute("aria-busy", "true");
    button.innerHTML = '<span class="spinner-border spinner-border-sm me-2" role="status" aria-hidden="true"></span>Predicting…';
    showResult(button, "Prediction in progress… Please wait for the model/API response.");
    window.setTimeout(() => {
      if (activeButtons.has(button)) restore(button,
        "The API is taking longer than expected. Check the Django terminal and browser Network tab for the backend error.");
    }, 90000);
  }, true);

  // Do not modify navigation tab labels or install a global fetch wrapper.
})();
/* DAY33_UI_LOADING_AUTOCOMPLETE_HELPER_END */
"""
helper = helper.replace("__SEARCH_URL__", json.dumps(search_url))
block = "<script>\n" + helper + "\n</script>\n"
idx = clean_html.lower().rfind("</body>")
clean_html = clean_html[:idx] + block + clean_html[idx:] if idx >= 0 else clean_html + "\n" + block
TEMPLATE.write_text(clean_html, encoding="utf-8", newline="")
print("Installed safe autocomplete/loading helper in", TEMPLATE)
print("Autocomplete URL:", search_url or "(not discovered)")
print("No navigation labels are modified; blank prediction requests are stopped before loading.")


Installed safe autocomplete/loading helper in D:\Git\Personal\ipl-player-performance\django_api\ipl_api\templates\ipl_api\dashboard_day25.html
Autocomplete URL: /api/players/search-options/
No navigation labels are modified; blank prediction requests are stopped before loading.


In [134]:
result = subprocess.run([sys.executable, "manage.py", "check"], cwd=DJANGO, capture_output=True, text=True, errors="replace")
print("python manage.py check exit code:", result.returncode)
print(result.stdout or "(no stdout)")
print(result.stderr or "(no stderr)")
print("\nNext: restart Django and hard-refresh the browser with Ctrl+Shift+R.")


python manage.py check exit code: 0
System check identified no issues (0 silenced).

(no stderr)

Next: restart Django and hard-refresh the browser with Ctrl+Shift+R.


## After running

1. Run all cells from the repository root in order.
2. Review the printed JavaScript path and discovered player-search URL.
3. Restart Django, then hard-refresh the dashboard with **Ctrl+Shift+R**.
4. Confirm player autocomplete options can be clicked, navigation labels remain unchanged, and prediction buttons show loading only while a prediction request is running.
5. Run `python manage.py check` from `django_api`.

This patch does not change model inference or database logic. It adds a frontend helper and preserves the existing prediction endpoints.
